# Painel do Emprego Formal — Geração e eliminação de empregos (Novo CAGED) · versão Python

**Vagas geradas** = admissões · **Vagas eliminadas** = desligamentos · **Saldo** = diferença · Fonte: Novo CAGED / MTE.

**Como usar**
1. Envie a planilha `Tabelas01.xlsx` (44,5 MB) pelo ícone de **pasta** (barra lateral esquerda) > **upload** e **espere a barra de progresso terminar**. Ela precisa ficar em `/content/Tabelas01.xlsx`.
2. Rode as células **na ordem** (`Shift+Enter`). Cada etapa tem uma célula de **conferência** (✔ / ✘).
3. No final, o painel é gravado em `/content/painel.html` e baixado.

Se a sessão reiniciar, o Colab apaga os arquivos: reenvie a planilha e rode tudo de novo.

## Etapa 0 — Instalar e configurar

In [ ]:
# geobr: malhas do IBGE para os mapas (o restante já vem instalado no Colab)
!pip -q install geobr

In [ ]:
import os, re, html, glob, zipfile, urllib.parse
import numpy as np
import pandas as pd

def confere(cond, msg):
    print('✔' if cond else '✘', msg)

# planilha enviada ao Colab; se esse nome não existir, usa o primeiro .xlsx da pasta /content
ARQUIVO = '/content/Tabelas01.xlsx'
if not os.path.exists(ARQUIVO):
    achados = sorted(glob.glob('/content/*.xlsx'))
    if achados:
        ARQUIVO = achados[0]

confere(os.path.exists(ARQUIVO), 'planilha encontrada em ' + ARQUIVO)
confere(zipfile.is_zipfile(ARQUIVO), 'arquivo .xlsx íntegro (se ✘, o upload não terminou: apague e envie de novo)')
if os.path.exists(ARQUIVO):
    print(os.path.getsize(ARQUIVO), 'bytes (o Tabelas01.xlsx completo tem 44543010)')

## Etapa 1 — Série mensal do Brasil (Tabela 5.1)
A aba é lida sem cabeçalho e o título **Mês** é localizado pelo conteúdo, não por posição fixa.

In [ ]:
MESES = {'Janeiro': 1, 'Fevereiro': 2, 'Março': 3, 'Abril': 4, 'Maio': 5, 'Junho': 6,
         'Julho': 7, 'Agosto': 8, 'Setembro': 9, 'Outubro': 10, 'Novembro': 11, 'Dezembro': 12}

def parse_mes(x):
    """'Janeiro/2020' -> Timestamp('2020-01-01'); qualquer outro texto -> NaT"""
    m = re.match(r'^([^/]+)/(\d{4})$', str(x).strip())
    if m and m.group(1) in MESES:
        return pd.Timestamp(int(m.group(2)), MESES[m.group(1)], 1)
    return pd.NaT

def ler_aba(aba):
    return pd.read_excel(ARQUIVO, sheet_name=aba, header=None, dtype=object).to_numpy(dtype=object)

def achar(m, texto):
    pos = np.argwhere(m == texto)
    if len(pos) == 0:
        raise ValueError(f'Texto não encontrado na aba: {texto}')
    return int(pos[0][0]), int(pos[0][1])

def num(valores):
    return pd.to_numeric(pd.Series(list(valores), dtype=object), errors='coerce').to_numpy(dtype=float)

def linhas_ate_vazio(m, inicio, col):
    linhas = []
    for r in range(inicio, m.shape[0]):
        if pd.isna(m[r, col]):
            break
        linhas.append(r)
    return linhas

def ler_brasil_mensal(aba='Tabela 5.1'):
    m = ler_aba(aba)
    i, j = achar(m, 'Mês')
    lin = linhas_ate_vazio(m, i + 1, j)
    d = pd.DataFrame({'rotulo': m[lin, j],
                      'estoque': num(m[lin, j + 1]),
                      'vagas_geradas': num(m[lin, j + 2]),
                      'vagas_eliminadas': num(m[lin, j + 3]),
                      'saldo': num(m[lin, j + 4])})
    d['mes'] = d['rotulo'].map(parse_mes)
    return d

brasil = ler_brasil_mensal()

tabela_principal = (brasil.sort_values('mes', ascending=False)
                    .rename(columns={'rotulo': 'Mês', 'vagas_geradas': 'Vagas geradas',
                                     'vagas_eliminadas': 'Vagas eliminadas', 'saldo': 'Saldo',
                                     'estoque': 'Estoque'})
                    [['Mês', 'Vagas geradas', 'Vagas eliminadas', 'Saldo', 'Estoque']])
brasil.tail(3)

In [ ]:
# CONFERÊNCIA Etapa 1
confere(len(brasil) == 80, '80 meses (Jan/2020 a Ago/2026)')
confere(brasil['rotulo'].iloc[-1] == 'Agosto/2026', 'último mês = Agosto/2026')
confere(brasil['saldo'].iloc[-1] == 165827, 'saldo de Agosto/2026 = 165.827')
confere(brasil['mes'].notna().all(), 'todos os meses foram convertidos em data')

## Etapa 2 — Bases longas por setor e por UF (Tabelas 6.1 e 7.1)
Cada mês ocupa um **bloco de colunas**. A função acha onde começa cada mês e empilha os blocos em uma base **longa**: uma linha por grupo e por mês.

In [ ]:
def ler_serie_larga(aba, rotulo):
    m = ler_aba(aba)
    i, jn = achar(m, rotulo)
    cab = m[i]
    jm = [j for j, v in enumerate(cab) if pd.notna(parse_mes(v))]      # colunas onde começa cada mês
    lin = linhas_ate_vazio(m, i + 2, jn)
    nomes = [re.sub(r'\s+', ' ', re.sub(r'\*+', '', str(x))).strip() for x in m[lin, jn]]
    partes = []
    for j in jm:
        partes.append(pd.DataFrame({'nome': nomes, 'mes': parse_mes(cab[j]),
                                    'estoque': num(m[lin, j]),
                                    'vagas_geradas': num(m[lin, j + 1]),
                                    'vagas_eliminadas': num(m[lin, j + 2]),
                                    'saldo': num(m[lin, j + 3])}))
    return pd.concat(partes, ignore_index=True)

base_setor = ler_serie_larga('Tabela 6.1', 'Grupamento de Atividades Econômicas e Seção CNAE 2.0')
base_uf    = ler_serie_larga('Tabela 7.1', 'Região e UF')

GRANDES = ['Agricultura, pecuária, produção florestal, pesca e aquicultura',
           'Indústria geral', 'Construção',
           'Comércio; reparação de veículos automotores e motocicletas', 'Serviços']

SIGLAS = {'Rondônia': 'RO', 'Acre': 'AC', 'Amazonas': 'AM', 'Roraima': 'RR', 'Pará': 'PA', 'Amapá': 'AP',
          'Tocantins': 'TO', 'Maranhão': 'MA', 'Piauí': 'PI', 'Ceará': 'CE', 'Rio Grande do Norte': 'RN',
          'Paraíba': 'PB', 'Pernambuco': 'PE', 'Alagoas': 'AL', 'Sergipe': 'SE', 'Bahia': 'BA',
          'Minas Gerais': 'MG', 'Espírito Santo': 'ES', 'Rio de Janeiro': 'RJ', 'São Paulo': 'SP',
          'Paraná': 'PR', 'Santa Catarina': 'SC', 'Rio Grande do Sul': 'RS', 'Mato Grosso do Sul': 'MS',
          'Mato Grosso': 'MT', 'Goiás': 'GO', 'Distrito Federal': 'DF'}

base_uf.head(3)

In [ ]:
# CONFERÊNCIA Etapa 2
ult = base_uf['mes'].max()
confere(base_uf['mes'].nunique() == 80, 'base de UFs com 80 meses')
confere(base_uf['nome'].iloc[0] == 'Brasil', 'primeira linha da base de UFs = Brasil')
confere(base_uf.query("nome == 'Brasil' and mes == @ult")['saldo'].iloc[0] == 165827, 'Brasil: saldo do último mês = 165.827')
confere(set(GRANDES) <= set(base_setor['nome']), 'os 5 grandes setores estão na base de setores')
primeiro = base_setor['nome'].iloc[0]
confere(base_setor.query("nome == @primeiro and mes == @ult")['saldo'].iloc[0] == 165827, 'total dos setores = saldo do Brasil')
confere(set(SIGLAS) <= set(base_uf['nome']), 'as 27 UFs estão na base de UFs')

## Etapa 3 — Tabelas de indicadores
- **Acumulado no ano:** soma do saldo desde janeiro, reiniciando a cada ano.
- **Acumulado em 12 meses:** soma do saldo do mês e dos 11 anteriores.
- **Variação contra 12 meses antes:** variação percentual do estoque em relação ao mesmo mês do ano anterior.

In [ ]:
def indicadores(base):
    ordem = {n: k for k, n in enumerate(dict.fromkeys(base['nome']))}     # mantém a ordem da planilha
    b = base.assign(_o=base['nome'].map(ordem)).sort_values(['_o', 'mes']).drop(columns='_o').reset_index(drop=True)
    b['ano'] = b['mes'].dt.year
    b['acum_ano'] = b.groupby(['nome', 'ano'], sort=False)['saldo'].cumsum()
    b['acum_12m'] = b.groupby('nome', sort=False)['saldo'].transform(lambda s: s.rolling(12).sum())
    b['var_12m']  = b.groupby('nome', sort=False)['estoque'].transform(lambda s: (s / s.shift(12) - 1) * 100)
    return b.drop(columns='ano')

ind_uf    = indicadores(base_uf)
ind_setor = indicadores(base_setor)

mes_ref   = ind_uf['mes'].max()
ult_br    = ind_uf[(ind_uf['nome'] == 'Brasil') & (ind_uf['mes'] == mes_ref)].iloc[0]
saldo_ano = ult_br['acum_ano']
saldo_12m = ult_br['acum_12m']
ult_br

In [ ]:
# CONFERÊNCIA Etapa 3
confere(saldo_ano == 1134033, 'saldo acumulado em 2026 = 1.134.033')
confere(saldo_12m == 887708, 'saldo em 12 meses = 887.708')

## Etapa 4 — Tabela final larga (3 últimas observações)
Uma linha por setor (ou UF) e, nas colunas, os 3 últimos meses de cada indicador.

In [ ]:
def tabela_larga(ind):
    ultimos3 = sorted(ind['mes'].dropna().unique())[-3:]
    d = ind[ind['mes'].isin(ultimos3)].copy()
    d['col'] = d['mes'].dt.strftime('%Y-%m')
    w = d.pivot(index='nome', columns='col', values=['saldo', 'acum_ano', 'acum_12m', 'var_12m'])
    w.columns = [f'{a}_{b}' for a, b in w.columns]
    ordem = list(dict.fromkeys(ind['nome']))
    return w.loc[ordem].reset_index()

final_setor = tabela_larga(ind_setor)
final_uf    = tabela_larga(ind_uf[ind_uf['nome'].isin(['Brasil'] + list(SIGLAS))])
list(final_setor.columns)

In [ ]:
# CONFERÊNCIA Etapa 4
confere(final_setor.shape[1] == 13, 'tabela final de setores: nome + 12 colunas')
confere(len(final_uf) == 28, 'tabela final de UFs: Brasil + 27 UFs')
confere(final_uf.loc[final_uf['nome'] == 'Brasil', 'acum_ano_2026-08'].iloc[0] == 1134033, 'Brasil: acumulado no ano em ago/2026')

## Etapa 5 — Índices trimestrais e semestrais
- **Variação relativa (%)** = saldo ÷ estoque do mês anterior ao início do período × 100
- **Índice de emprego** = estoque ao fim do período ÷ estoque anterior ao 1º período × 100 (base 100)
- **Taxas** de admissão e desligamento = vagas ÷ estoque inicial × 100
- **Rotatividade** = (geradas + eliminadas) ÷ 2 ÷ estoque inicial × 100
- `parcial = True` marca o período ainda incompleto (o 3T/2026 só tem jul–ago).

In [ ]:
def agrega(df, mpp, grupo=None):
    """mpp = meses por período (3 = trimestre, 6 = semestre)"""
    chaves = list(grupo) if grupo else ['_g']
    d = df.assign(_g=1, ano=df['mes'].dt.year, per=(df['mes'].dt.month - 1) // mpp + 1)
    d = d.sort_values(chaves + ['mes'])
    r = (d.groupby(chaves + ['ano', 'per'], sort=True)
           .agg(meses=('mes', 'size'),
                vagas_geradas=('vagas_geradas', 'sum'),
                vagas_eliminadas=('vagas_eliminadas', 'sum'),
                saldo=('saldo', 'sum'),
                _est_prim=('estoque', 'first'), _sal_prim=('saldo', 'first'),
                estoque_fim=('estoque', 'last'))
           .reset_index())
    r['estoque_ini'] = r['_est_prim'] - r['_sal_prim']            # estoque do mês anterior
    base = r.groupby(chaves)['estoque_ini'].transform('first')
    sfx = 'T' if mpp == 3 else 'S'
    r['periodo'] = r['per'].astype(str) + sfx + '/' + r['ano'].astype(str)
    r['inicio'] = pd.to_datetime(dict(year=r['ano'], month=(r['per'] - 1) * mpp + 1, day=1))
    r['parcial'] = r['meses'] < mpp
    r['var_pct'] = r['saldo'] / r['estoque_ini'] * 100
    r['indice'] = r['estoque_fim'] / base * 100
    r['taxa_adm'] = r['vagas_geradas'] / r['estoque_ini'] * 100
    r['taxa_desl'] = r['vagas_eliminadas'] / r['estoque_ini'] * 100
    r['rotatividade'] = (r['vagas_geradas'] + r['vagas_eliminadas']) / 2 / r['estoque_ini'] * 100
    k = 12 // mpp
    g = r.groupby(chaves, sort=False)
    r['saldo_12m'] = g['saldo'].transform(lambda s: s.rolling(k).sum())
    r['var_12m_pct'] = r['saldo_12m'] / g['estoque_ini'].shift(k - 1) * 100
    r.loc[r['parcial'], ['saldo_12m', 'var_12m_pct']] = np.nan     # janela incompleta
    return r.drop(columns=['_est_prim', '_sal_prim', '_g'], errors='ignore')

tri = agrega(brasil, 3)
sem = agrega(brasil, 6)
tri_setor = agrega(base_setor[base_setor['nome'].isin(GRANDES)], 3, ['nome'])
sem_setor = agrega(base_setor[base_setor['nome'].isin(GRANDES)], 6, ['nome'])
tri_uf    = agrega(base_uf[base_uf['nome'].isin(SIGLAS)], 3, ['nome'])
sem_uf    = agrega(base_uf[base_uf['nome'].isin(SIGLAS)], 6, ['nome'])

tri[['periodo', 'meses', 'saldo', 'var_pct', 'parcial']].tail(4)

In [ ]:
# CONFERÊNCIA Etapa 5
def v(d, p, col):
    return d.loc[d['periodo'] == p, col].iloc[0]

confere(v(tri, '1T/2026', 'saldo') == 614076, '1T/2026: saldo 614.076')
confere(v(tri, '2T/2026', 'saldo') == 298215, '2T/2026: saldo 298.215')
confere(v(tri, '3T/2026', 'saldo') == 221742 and bool(v(tri, '3T/2026', 'parcial')), '3T/2026: saldo 221.742 (parcial)')
confere(v(sem, '1S/2026', 'saldo') == 912291, '1S/2026: saldo 912.291')
confere(abs(v(tri, '1T/2026', 'var_pct') - 1.303) < 0.01, '1T/2026: variação de 1,30%')

## Etapa 6 — Gráficos (plotly)

In [ ]:
import plotly.graph_objects as go

COR_POS, COR_NEG, COR_AZUL = '#2e7d32', '#c62828', '#1565c0'

def ajusta(fig, altura=360):
    fig.update_layout(separators=',.', height=altura, template='plotly_white',
                      margin=dict(l=55, r=20, t=30, b=45),
                      legend=dict(orientation='h', y=-0.2))
    return fig

def rotulos(df):
    return [p + ('*' if parc else '') for p, parc in zip(df['periodo'], df['parcial'])]

def fig_geracao(df):
    f = go.Figure()
    f.add_scatter(x=df['mes'], y=df['vagas_geradas'], name='Vagas geradas', line=dict(color=COR_POS))
    f.add_scatter(x=df['mes'], y=df['vagas_eliminadas'], name='Vagas eliminadas', line=dict(color=COR_NEG))
    return ajusta(f)

def fig_saldo(df):
    cores = [COR_POS if s >= 0 else COR_NEG for s in df['saldo']]
    f = go.Figure(go.Bar(x=df['mes'], y=df['saldo'], marker_color=cores, name='Saldo'))
    return ajusta(f)

def fig_periodo_barras(df, col, titulo_y):
    cores = [COR_POS if (pd.isna(s) or s >= 0) else COR_NEG for s in df[col]]
    f = go.Figure(go.Bar(x=rotulos(df), y=df[col], marker_color=cores))
    f.update_xaxes(categoryorder='array', categoryarray=rotulos(df))
    f.update_yaxes(title=titulo_y)
    return ajusta(f)

def fig_indice(df):
    f = go.Figure(go.Scatter(x=rotulos(df), y=df['indice'], mode='lines+markers', line=dict(color=COR_AZUL)))
    f.update_xaxes(categoryorder='array', categoryarray=rotulos(df))
    f.update_yaxes(title='Índice (base 100 = estoque pré-2020)')
    return ajusta(f)

def fig_taxas(df):
    f = go.Figure()
    f.add_scatter(x=rotulos(df), y=df['taxa_adm'], name='Taxa de admissão (%)', line=dict(color=COR_POS))
    f.add_scatter(x=rotulos(df), y=df['taxa_desl'], name='Taxa de desligamento (%)', line=dict(color=COR_NEG))
    f.update_xaxes(categoryorder='array', categoryarray=rotulos(df))
    f.update_yaxes(title='% do estoque inicial')
    return ajusta(f)

def fig_setor(ind_setor, mes):
    d = ind_setor[(ind_setor['mes'] == mes) & (ind_setor['nome'].isin(GRANDES))].sort_values('saldo')
    cores = [COR_POS if s >= 0 else COR_NEG for s in d['saldo']]
    f = go.Figure(go.Bar(x=d['saldo'], y=d['nome'], orientation='h', marker_color=cores))
    return ajusta(f, 320)

g_geracao = fig_geracao(brasil)
g_saldo   = fig_saldo(brasil)
g_tri     = fig_periodo_barras(tri, 'saldo', 'Saldo de vagas')
g_tri_var = fig_periodo_barras(tri, 'var_pct', '% do estoque inicial')
g_sem     = fig_periodo_barras(sem, 'saldo', 'Saldo de vagas')
g_sem_var = fig_periodo_barras(sem, 'var_pct', '% do estoque inicial')
g_ind_tri = fig_indice(tri)
g_ind_sem = fig_indice(sem)
g_tax_tri = fig_taxas(tri)
g_tax_sem = fig_taxas(sem)
g_setor   = fig_setor(ind_setor, mes_ref)

confere(all(isinstance(g, go.Figure) for g in [g_geracao, g_saldo, g_tri, g_sem, g_ind_tri, g_setor]), 'gráficos criados')
g_geracao      # no Colab, o gráfico aparece aqui

## Etapa 7 — Mapas por UF (folium)
A planilha traz o **nome** da UF e a malha do IBGE usa a **sigla**; por isso a tabela `SIGLAS`. A escala é divergente e simétrica em torno de zero: verde = saldo positivo, vermelho = negativo.

In [ ]:
import geobr, folium
from branca.colormap import linear

uf_ult = ind_uf[(ind_uf['mes'] == mes_ref) & (ind_uf['nome'].isin(SIGLAS))].copy()
uf_ult['sigla'] = uf_ult['nome'].map(SIGLAS)

malha = geobr.read_state(year=2020)          # baixa a malha do IBGE (precisa de internet)
mapa_df = malha.merge(uf_ult[['sigla', 'nome', 'saldo', 'acum_ano', 'acum_12m', 'var_12m']],
                      left_on='abbrev_state', right_on='sigla', how='left')

confere(len(uf_ult) == 27, '27 UFs na base do mapa')
confere(mapa_df['saldo'].notna().all(), 'todas as UFs casaram com a malha do IBGE')

In [ ]:
def fazer_mapa(var, titulo):
    m = mapa_df[['nome', var, 'geometry']].copy()
    lim = float(m[var].abs().max())
    cmap = linear.RdYlGn_11.scale(-lim, lim)
    cmap.caption = titulo
    mp = folium.Map(location=[-14, -52], zoom_start=4, tiles='CartoDB positron')

    def estilo(feat):
        val = feat['properties'][var]
        cor = cmap(val) if (val is not None and val == val) else '#d9d9d9'
        return {'fillColor': cor, 'color': 'white', 'weight': 0.6, 'fillOpacity': 0.85}

    folium.GeoJson(m, style_function=estilo,
                   tooltip=folium.GeoJsonTooltip(fields=['nome', var], aliases=['UF', titulo], localize=True)).add_to(mp)
    cmap.add_to(mp)
    return mp

mapas = {
    'Saldo no mês':            fazer_mapa('saldo',    'Saldo (mês)'),
    'Saldo no ano':            fazer_mapa('acum_ano', 'Saldo no ano'),
    'Saldo em 12 meses':       fazer_mapa('acum_12m', 'Saldo 12 meses'),
    'Variação em 12 meses (%)': fazer_mapa('var_12m',  'Variação 12 meses (%)'),
}
confere(len(mapas) == 4, 'mapas criados')
mapas['Saldo no mês']      # no Colab, o mapa aparece aqui

## Etapa 8 — Montar o painel (HTML)
Gera **um único arquivo** `painel.html` com 6 páginas (Apresentação, Indicadores, Índices trimestrais, Índices semestrais, Mapas e Dados).
As tabelas usam DataTables (busca e ordenação) carregado da internet; sem internet, elas aparecem como tabelas simples.

In [ ]:
from plotly.offline import get_plotlyjs

def fmt_int(x):
    return f'{x:,.0f}'.replace(',', '.')

def fmt_dec(x, d=2):
    return f'{x:,.{d}f}'.replace(',', '#').replace('.', ',').replace('#', '.')

def fig_html(fig):
    return fig.to_html(full_html=False, include_plotlyjs=False, config={'displaylogo': False, 'responsive': True})

def mapa_html(mp):
    return ('<iframe class="mapa" srcdoc="' + html.escape(mp.get_root().render(), quote=True) + '"></iframe>')

def csv_link(df, nome):
    txt = df.to_csv(sep=';', decimal=',', index=False)
    return (f'<a class="baixar" download="{nome}" '
            f'href="data:text/csv;charset=utf-8,{urllib.parse.quote(chr(65279) + txt)}">Baixar CSV</a>')

def tabela_html(df, tid, ints=(), decs=()):
    d = df.copy()
    cols = list(d.columns)
    for c in decs:
        d[c] = d[c].round(2)
    idx_int = ','.join(str(cols.index(c)) for c in ints)
    idx_dec = ','.join(str(cols.index(c)) for c in decs)
    t = d.to_html(index=False, border=0, classes='tabela display', table_id=tid, na_rep='', escape=True)
    t = t.replace('<table ', f'<table data-int="{idx_int}" data-dec="{idx_dec}" ', 1)
    return csv_link(df, tid + '.csv') + t

def card(titulo, conteudo, extra=''):
    return f'<section class="card {extra}"><h3>{titulo}</h3>{conteudo}</section>'

def kpi(rotulo, valor, cor):
    return f'<div class="kpi {cor}"><span>{rotulo}</span><strong>{valor}</strong></div>'

def periodo_tab(p, nome_periodo):
    cols = ['periodo', 'meses', 'vagas_geradas', 'vagas_eliminadas', 'saldo', 'estoque_fim',
            'var_pct', 'var_12m_pct', 'indice', 'taxa_adm', 'taxa_desl', 'rotatividade']
    d = p.sort_values('inicio', ascending=False)[cols].copy()
    d['periodo'] = [x + ('*' if parc else '') for x, parc in zip(d['periodo'], p.sort_values('inicio', ascending=False)['parcial'])]
    d.columns = [nome_periodo, 'Meses', 'Vagas geradas', 'Vagas eliminadas', 'Saldo', 'Estoque final',
                 'Variação (%)', 'Variação 12 meses (%)', 'Índice (base 100)', 'Taxa admissão (%)',
                 'Taxa desligamento (%)', 'Rotatividade (%)']
    return d

lab_mes = mes_ref.strftime('%m/%Y')

pag_apres = card('Sobre o painel', f"""
<p>Painel sobre a <b>geração e a eliminação de empregos formais</b> no Brasil (fonte: <b>Novo CAGED / MTE</b>).
<b>Vagas geradas</b> = admissões · <b>vagas eliminadas</b> = desligamentos · <b>saldo</b> = diferença.</p>
<p>Competência de referência: <b>{lab_mes}</b>. Saldo no mês: <b>{fmt_int(ult_br['saldo'])}</b>.
No ano: <b>{fmt_int(saldo_ano)}</b>. Em 12 meses: <b>{fmt_int(saldo_12m)}</b>.</p>""") + card('Metodologia', """
<ul><li><b>Variação relativa</b> = saldo do período ÷ estoque do mês anterior ao início do período × 100.</li>
<li><b>Índice de emprego</b> = estoque ao fim do período ÷ estoque pré-2020 × 100.</li>
<li><b>Taxas</b> = vagas geradas (ou eliminadas) ÷ estoque inicial × 100.</li>
<li><b>Rotatividade</b> = (geradas + eliminadas) ÷ 2 ÷ estoque inicial × 100.</li>
<li>Períodos marcados com * são parciais (ainda não têm todos os meses).</li></ul>""")

pag_ind = ('<div class="kpis">' + kpi('Saldo (mês)', fmt_int(ult_br['saldo']), 'azul')
           + kpi('Saldo no ano', fmt_int(saldo_ano), 'verde')
           + kpi('Saldo em 12 meses', fmt_int(saldo_12m), 'verde')
           + kpi('Estoque de empregos', fmt_int(ult_br['estoque']), 'cinza') + '</div>'
           + '<div class="grade">' + card('Vagas geradas e eliminadas por mês', fig_html(g_geracao))
           + card('Saldo mensal de vagas', fig_html(g_saldo)) + '</div>'
           + card('Saldo por grande setor (mês de referência)', fig_html(g_setor)))

def pagina_periodo(g_saldo_p, g_var_p, g_ind_p, g_tax_p, p, p_setor, p_uf, nome, tag):
    return ('<div class="grade">' + card(f'Saldo por {nome} (* = parcial)', fig_html(g_saldo_p))
            + card(f'Variação relativa no {nome} (%)', fig_html(g_var_p)) + '</div>'
            + '<div class="grade">' + card('Índice de emprego (base 100 = pré-2020)', fig_html(g_ind_p))
            + card('Taxas de admissão e de desligamento', fig_html(g_tax_p)) + '</div>'
            + card(f'Brasil · índices por {nome}', tabela_html(
                periodo_tab(p, nome.capitalize()), f'br_{tag}',
                ints=['Vagas geradas', 'Vagas eliminadas', 'Saldo', 'Estoque final'],
                decs=['Variação (%)', 'Variação 12 meses (%)', 'Índice (base 100)', 'Taxa admissão (%)',
                      'Taxa desligamento (%)', 'Rotatividade (%)']))
            + card('Por grande setor', tabela_html(
                p_setor.sort_values(['inicio', 'nome'], ascending=[False, True])
                       [['periodo', 'nome', 'vagas_geradas', 'vagas_eliminadas', 'saldo', 'var_pct', 'var_12m_pct', 'indice']]
                       .set_axis([nome.capitalize(), 'Setor', 'Vagas geradas', 'Vagas eliminadas', 'Saldo',
                                  'Variação (%)', 'Variação 12 meses (%)', 'Índice (base 100)'], axis=1),
                f'setor_{tag}', ints=['Vagas geradas', 'Vagas eliminadas', 'Saldo'],
                decs=['Variação (%)', 'Variação 12 meses (%)', 'Índice (base 100)']))
            + card('Por UF', tabela_html(
                p_uf.sort_values(['inicio', 'nome'], ascending=[False, True])
                    [['periodo', 'nome', 'vagas_geradas', 'vagas_eliminadas', 'saldo', 'var_pct', 'var_12m_pct', 'indice']]
                    .set_axis([nome.capitalize(), 'UF', 'Vagas geradas', 'Vagas eliminadas', 'Saldo',
                               'Variação (%)', 'Variação 12 meses (%)', 'Índice (base 100)'], axis=1),
                f'uf_{tag}', ints=['Vagas geradas', 'Vagas eliminadas', 'Saldo'],
                decs=['Variação (%)', 'Variação 12 meses (%)', 'Índice (base 100)'])))

pag_tri = pagina_periodo(g_tri, g_tri_var, g_ind_tri, g_tax_tri, tri, tri_setor, tri_uf, 'trimestre', 'tri')
pag_sem = pagina_periodo(g_sem, g_sem_var, g_ind_sem, g_tax_sem, sem, sem_setor, sem_uf, 'semestre', 'sem')

pag_mapas = '<div class="grade">' + ''.join(card(t, mapa_html(m)) for t, m in mapas.items()) + '</div>'

pag_dados = (card('Geração e eliminação de empregos (Brasil, por mês)',
                  tabela_html(tabela_principal, 'principal',
                              ints=['Vagas geradas', 'Vagas eliminadas', 'Saldo', 'Estoque']))
             + card('Grandes setores e seções · 3 últimos meses', tabela_html(final_setor, 'final_setor',
                    decs=[c for c in final_setor.columns if c != 'nome']))
             + card('UFs · 3 últimos meses', tabela_html(final_uf, 'final_uf',
                    decs=[c for c in final_uf.columns if c != 'nome'])))

PAGINAS = [('apres', 'Apresentação', pag_apres), ('ind', 'Indicadores', pag_ind),
           ('tri', 'Índices trimestrais', pag_tri), ('sem', 'Índices semestrais', pag_sem),
           ('mapas', 'Mapas', pag_mapas), ('dados', 'Dados', pag_dados)]
confere(len(PAGINAS) == 6, '6 páginas montadas')

In [ ]:
MODELO = """<!DOCTYPE html>
<html lang="pt-BR"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Painel do Emprego Formal</title>
<link rel="stylesheet" href="https://cdn.datatables.net/1.13.6/css/jquery.dataTables.min.css">
<style>
 body{font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;margin:0;background:#f4f6f8;color:#222}
 header{background:#1b3a57;color:#fff;padding:14px 22px}
 header h1{margin:0;font-size:20px} header p{margin:2px 0 0;font-size:13px;opacity:.85}
 nav{background:#fff;border-bottom:1px solid #d9dee3;position:sticky;top:0;z-index:5;padding:0 14px;overflow-x:auto;white-space:nowrap}
 nav button{border:0;background:none;padding:12px 14px;font-size:14px;cursor:pointer;color:#455;border-bottom:3px solid transparent}
 nav button.ativo{color:#1b3a57;border-bottom-color:#1b3a57;font-weight:600}
 main{padding:16px 22px}
 .pagina{display:none}
 .grade{display:grid;grid-template-columns:repeat(auto-fit,minmax(420px,1fr));gap:14px}
 .card{background:#fff;border-radius:8px;padding:12px 16px;margin-bottom:14px;box-shadow:0 1px 3px rgba(0,0,0,.08);overflow-x:auto}
 .card h3{margin:0 0 8px;font-size:15px;color:#1b3a57}
 .kpis{display:grid;grid-template-columns:repeat(auto-fit,minmax(200px,1fr));gap:14px;margin-bottom:14px}
 .kpi{border-radius:8px;padding:14px 16px;color:#fff}
 .kpi span{display:block;font-size:13px;opacity:.9} .kpi strong{font-size:26px}
 .kpi.azul{background:#1565c0}.kpi.verde{background:#2e7d32}.kpi.cinza{background:#546e7a}
 iframe.mapa{width:100%;height:460px;border:0}
 table.tabela{width:100%;font-size:13px} a.baixar{display:inline-block;margin-bottom:8px;font-size:13px}
</style>
<script>__PLOTLYJS__</script>
</head><body>
<header><h1>Painel do Emprego Formal</h1>
<p>Geração e eliminação de empregos · Novo CAGED / MTE · competência __COMPETENCIA__</p></header>
<nav>__NAV__</nav>
<main>__PAGINAS__</main>
<script src="https://code.jquery.com/jquery-3.7.1.min.js"></script>
<script src="https://cdn.datatables.net/1.13.6/js/jquery.dataTables.min.js"></script>
<script>
function abrir(id){
  document.querySelectorAll('.pagina').forEach(function(p){p.style.display='none';});
  document.getElementById(id).style.display='block';
  document.querySelectorAll('nav button').forEach(function(b){b.classList.toggle('ativo', b.dataset.alvo===id);});
  document.querySelectorAll('#'+id+' .plotly-graph-div').forEach(function(d){ if(window.Plotly) Plotly.Plots.resize(d); });
  if(window.jQuery && jQuery.fn.dataTable){ jQuery.fn.dataTable.tables({visible:true, api:true}).columns.adjust(); }
}
window.addEventListener('load', function(){
  if(window.jQuery && jQuery.fn.dataTable){
    var pt={search:'Buscar:', lengthMenu:'Mostrar _MENU_', info:'_START_ a _END_ de _TOTAL_', infoEmpty:'Sem registros',
            paginate:{previous:'Anterior', next:'Próximo'}, zeroRecords:'Nada encontrado'};
    var idx=function(v){return (v+'').split(',').filter(String).map(Number);};
    jQuery('table.tabela').each(function(){
      var t=jQuery(this), ints=idx(t.attr('data-int')), decs=idx(t.attr('data-dec')), defs=[];
      if(ints.length) defs.push({targets:ints, render:function(d,tp){return (tp==='display' && d!=='')? Number(d).toLocaleString('pt-BR',{maximumFractionDigits:0}) : d;}});
      if(decs.length) defs.push({targets:decs, render:function(d,tp){return (tp==='display' && d!=='')? Number(d).toLocaleString('pt-BR',{minimumFractionDigits:2,maximumFractionDigits:2}) : d;}});
      t.DataTable({pageLength:12, scrollX:true, columnDefs:defs, language:pt});
    });
  }
  abrir('__PRIMEIRA__');
});
</script></body></html>"""

nav = ''.join(f'<button data-alvo="{i}" onclick="abrir(\'{i}\')">{t}</button>' for i, t, _ in PAGINAS)
corpo = ''.join(f'<div class="pagina" id="{i}">{c}</div>' for i, _, c in PAGINAS)

saida = (MODELO.replace('__COMPETENCIA__', lab_mes).replace('__NAV__', nav)
               .replace('__PAGINAS__', corpo).replace('__PRIMEIRA__', PAGINAS[0][0])
               .replace('__PLOTLYJS__', get_plotlyjs()))

with open('/content/painel.html', 'w', encoding='utf-8') as f:
    f.write(saida)

confere(os.path.exists('/content/painel.html'), f"painel.html gerado ({os.path.getsize('/content/painel.html') / 1e6:.1f} MB)")

In [ ]:
# Baixar o painel para o seu computador (abra o arquivo no navegador)
from google.colab import files
files.download('/content/painel.html')

## Para evoluir
- **Mapa por município:** leia a Tabela 3 e junte pelo código IBGE de 6 dígitos (use os 6 primeiros dígitos de `code_muni` em `geobr.read_municipality`). O arquivo fica pesado.
- **Novo mês:** envie a nova planilha e rode tudo de novo. As conferências com valores fixos (como 165.827) mostrarão ✘, porque são de agosto/2026; troque pelos valores do novo mês ou apague essas linhas.

## Se algo der errado
| Sintoma | Causa provável |
|---|---|
| `FileNotFoundError` | Planilha não enviada ou sessão reiniciada |
| `ModuleNotFoundError: geobr` | A célula da Etapa 0 (`pip install geobr`) não rodou |
| ✘ na conferência de UFs | Nome de UF da planilha diferente do dicionário `SIGLAS` |
| `KeyError` ou `IndexError` ao ler a aba | A planilha mudou de layout; confira o título usado em `achar(...)` |
| Painel sem formatação nas tabelas | Sem internet para carregar o DataTables (as tabelas continuam funcionando) |